# 08C – Probability Calibration

Enterprise notebook for evaluating whether predicted bankruptcy probabilities are well calibrated.

## Business Objective
Assess whether the predicted probabilities accurately represent the true likelihood of bankruptcy and improve them if necessary.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import RandomForestClassifier

In [ ]:
DATA_PATH='american_bankruptcy_cleaned.csv'
MODEL_PATH='production_bankruptcy_model.joblib'

df=pd.read_csv(DATA_PATH)

if 'status_label' in df.columns:
    y=df['status_label'].map({'alive':0,'failed':1})
    X=df.drop(columns=['status_label'])
elif 'target' in df.columns:
    y=df['target']
    X=df.drop(columns=['target'])
else:
    raise ValueError('Target column not found')

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)

model=joblib.load(MODEL_PATH)
probs=model.predict_proba(X_test)[:,1]

In [ ]:
frac_pos, mean_pred = calibration_curve(
    y_test,
    probs,
    n_bins=10,
    strategy='quantile'
)

brier = brier_score_loss(y_test, probs)
print(f'Brier Score: {brier:.4f}')

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(mean_pred, frac_pos, marker='o', label='Model')
plt.plot([0,1],[0,1],'--',label='Perfect Calibration')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Frequency')
plt.title('Calibration Curve')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('calibration_curve.png', dpi=300)
plt.show()

In [ ]:
pd.DataFrame({
    'Mean_Predicted_Probability': mean_pred,
    'Observed_Frequency': frac_pos
}).to_csv('calibration_results.csv', index=False)

## Business Interpretation

- A curve close to the diagonal indicates well-calibrated probabilities.
- Curves above the diagonal indicate underestimated risk.
- Curves below the diagonal indicate overestimated risk.
- Use the Brier Score to compare calibration quality across models.

## Deliverables

- `calibration_curve.png`
- `calibration_results.csv`

This notebook evaluates the reliability of predicted probabilities, an important requirement for production risk scoring systems.